# SleepAgent Evidence Extraction v1.2 Debug

单步测试 Phase 1 Grounding 的 Evidence Extraction v1.2。

默认行为：不访问网络，不调用真实 LLM，不修改 raw data。pipeline 写 outputs 的步骤放在后面独立 cell。

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "sleep_ai_scientist").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)
print("package exists =", (ROOT / "sleep_ai_scientist").exists())

ROOT = /home/jwj/code/SleepAgent
package exists = True


## 1. Load configs and rules

In [2]:
from sleep_ai_scientist.common.config import load_config, config_path
from sleep_ai_scientist.common.io import read_yaml
from sleep_ai_scientist.grounding.evidence_extractor import load_evidence_rules

GROUNDING_CONFIG = ROOT / "configs/grounding_config.yaml"
RULES_CONFIG = ROOT / "configs/evidence_extraction_rules.yaml"
LLM_CONFIG = ROOT / "configs/llm_config.yaml"

config = load_config(GROUNDING_CONFIG)
rules = load_evidence_rules(RULES_CONFIG)
llm_config = read_yaml(LLM_CONFIG)

print("mechanism_count =", len(rules["mechanisms"]))
print("animal species rules =", sorted(rules["animal_terms"]["species"].keys()))
print("llm enabled =", llm_config["llm"]["enabled"])
print("llm assist enabled =", llm_config["evidence_extraction"]["llm_assist_enabled"])

mechanism_count = 21
animal species rules = ['cat', 'mouse', 'nonhuman_primate', 'rat']
llm enabled = False
llm assist enabled = False


## 2. Inspect literature query set

这里检查 `configs/literature_queries.yaml`，确认 query set version、provider、query 数量。

In [ ]:
from sleep_ai_scientist.api.literature_client import apply_query_config

QUERY_CONFIG = ROOT / "configs/literature_queries.yaml"
query_payload = read_yaml(QUERY_CONFIG)
query_groups = query_payload.get("queries", {})
all_queries = [q for values in query_groups.values() for q in (values or [])]

print("query_set_version =", query_payload.get("query_set", {}).get("version"))
print("groups =", list(query_groups.keys()))
print("query_count =", len(all_queries))
print("providers =", query_payload.get("settings", {}).get("providers"))
print("max_results_per_query =", query_payload.get("settings", {}).get("max_results_per_query"))
print("first 5 queries:")
for q in all_queries[:5]:
    print("-", q)

## 3. Optional API paper download

默认不运行。需要真实下载论文时，把 `RUN_API_DOWNLOAD = True`。

建议先限制 `DEBUG_QUERIES` 和 `DEBUG_PROVIDERS`，避免一次下载完整 v1 corpus。

In [ ]:
from copy import deepcopy
from sleep_ai_scientist.api.literature_client import search_literature_apis, load_env_file

RUN_API_DOWNLOAD = False
DEBUG_QUERIES = ["insomnia slow wave EEG"]
DEBUG_PROVIDERS = ["openalex"]
DEBUG_MAX_RESULTS_PER_QUERY = 5

api_config = apply_query_config(deepcopy(config), QUERY_CONFIG)
api_config.setdefault("api", {})["enabled"] = True
api_config["api"]["search_queries"] = DEBUG_QUERIES
api_config["api"]["max_results_per_query"] = DEBUG_MAX_RESULTS_PER_QUERY
for provider, provider_cfg in api_config["api"].get("providers", {}).items():
    provider_cfg["enabled"] = provider in DEBUG_PROVIDERS

loaded_env = load_env_file(ROOT / ".env")
print("loaded .env keys =", loaded_env)
print("download queries =", api_config["api"]["search_queries"])
print("enabled providers =", [p for p, c in api_config["api"].get("providers", {}).items() if c.get("enabled")])

if RUN_API_DOWNLOAD:
    api_papers, api_summary = search_literature_apis(api_config)
    print(json.dumps(api_summary, indent=2, ensure_ascii=False))
    print("downloaded api papers =", len(api_papers))
    for paper in api_papers[:5]:
        print("-", paper.paper_id, paper.year, paper.title[:120])
else:
    api_papers, api_summary = [], {"enabled": False, "note": "Set RUN_API_DOWNLOAD=True to download papers."}
    print(api_summary["note"])

## 4. Read downloaded papers from disk

如果你已经运行过 API build 或上一个下载 cell，这里读取 `api_retrieved_papers.csv/jsonl`。

In [ ]:
from sleep_ai_scientist.common.io import read_csv
from sleep_ai_scientist.grounding.literature_loader import normalize_literature_row

api_csv = ROOT / "data/literature/api_retrieved_papers.csv"
api_jsonl = ROOT / "data/literature/api_retrieved_papers.jsonl"
registry_csv = ROOT / "data/literature/literature_registry.csv"

print("api_csv exists =", api_csv.exists())
print("api_jsonl exists =", api_jsonl.exists())
print("registry_csv exists =", registry_csv.exists())

api_rows = read_csv(api_csv) if api_csv.exists() else []
print("api csv rows =", len(api_rows))
if api_rows:
    print("api csv columns =", list(api_rows[0].keys()))
    for row in api_rows[:3]:
        print("-", row.get("provider"), row.get("year"), row.get("title", "")[:120])

registry_rows = read_csv(registry_csv) if registry_csv.exists() else []
print("registry rows =", len(registry_rows))

## 5. Merge seed + API papers in memory

用于单步验证 dedup 逻辑。这个 cell 不写正式 registry 文件。

In [ ]:
from sleep_ai_scientist.grounding.literature_loader import load_literature
from sleep_ai_scientist.grounding.literature_registry import merge_literature_registry

seed_path = ROOT / "data/fixtures/toy_seed_papers.csv"
seed_papers = load_literature(seed_path)

disk_api_papers = []
if api_rows:
    disk_api_papers = [normalize_literature_row(row) for row in api_rows]

api_candidates = api_papers or disk_api_papers
merged_papers, duplicate_report = merge_literature_registry(seed_papers, api_candidates)

print("seed_papers =", len(seed_papers))
print("api_candidates =", len(api_candidates))
print("merged_papers =", len(merged_papers))
print("duplicate_groups =", len(duplicate_report))
for item in duplicate_report[:5]:
    print(item)

papers = merged_papers
print("papers variable now points to merged_papers")

## 6. Inspect extraction rules

逐项查看 mechanism keywords、variable keywords、modalities、animal terms 和 direction terms。

In [ ]:
mechanisms_to_check = [
    "slow-wave generation",
    "spindle generation",
    "hyperarousal",
    "orexin_hypocretin_arousal",
    "gabaergic_sleep_promotion",
    "glymphatic_clearance",
]

for mechanism in mechanisms_to_check:
    rule = rules["mechanisms"].get(mechanism, {})
    print("\n##", mechanism)
    print("mechanism_keywords =", rule.get("mechanism_keywords"))
    print("variable_keywords =", rule.get("variable_keywords"))
    print("modalities =", rule.get("modalities"))
    print("context_default =", rule.get("evidence_context_default"))
    print("downstream_role_default =", rule.get("downstream_role_default"))

print("\ndirection_terms =", rules.get("direction_terms"))
print("\nanimal_terms =", rules.get("animal_terms"))

In [ ]:
from sleep_ai_scientist.grounding.evidence_extractor import split_sentences, infer_direction, infer_species, infer_model_system

test_text = "Optogenetic stimulation of orexin neurons in mice increased wakefulness. Spindle density showed no significant association with insomnia severity."
for idx, sentence in enumerate(split_sentences(test_text)):
    species, species_terms = infer_species(sentence, rules)
    model_system, context, model_terms = infer_model_system(sentence, rules, species)
    print("sentence", idx, sentence)
    print("  direction =", infer_direction(sentence, rules).value)
    print("  species =", species, species_terms)
    print("  model/context =", model_system, context, model_terms)

## 7. Load seed papers only

In [ ]:
from sleep_ai_scientist.grounding.literature_loader import load_literature

USE_SEED_ONLY = False
seed_path = ROOT / "data/fixtures/toy_seed_papers.csv"
seed_only_papers = load_literature(seed_path)
if USE_SEED_ONLY:
    papers = seed_only_papers
elif "papers" not in globals():
    papers = seed_only_papers

print("seed_only_papers =", len(seed_only_papers))
print("papers =", len(papers))
for paper in papers:
    print("-", paper.paper_id, paper.title)

## 8. Sentence-level rule extraction

In [ ]:
from sleep_ai_scientist.grounding.evidence_extractor import extract_evidence

raw_evidence = extract_evidence(
    papers,
    default_population=config.get("evidence", {}).get("default_population", ""),
    rules_path=RULES_CONFIG,
)
print("raw_evidence =", len(raw_evidence))
for item in raw_evidence[:10]:
    print(item.evidence_id, "|", item.mechanism, "|", item.direction.value, "|", item.source_text)

In [ ]:
def evidence_rows(evidence):
    return [
        {
            "paper_id": e.paper_id,
            "mechanism": e.mechanism,
            "direction": e.direction.value,
            "variable": e.variable_or_feature,
            "species": e.species,
            "context": e.evidence_context,
            "downstream_role": e.downstream_role,
            "method": e.extraction_method,
            "matched_terms": e.matched_terms,
            "source_text": e.source_text,
        }
        for e in evidence
    ]

try:
    import pandas as pd
    display(pd.DataFrame(evidence_rows(raw_evidence)))
except Exception:
    print(json.dumps(evidence_rows(raw_evidence), indent=2, ensure_ascii=False))

## 9. Direction sanity checks

`no significant` / `no difference` 不应被判为 support。

In [ ]:
from sleep_ai_scientist.schemas.literature import LiteratureRecord

direction_paper = LiteratureRecord(
    paper_id="direction_demo",
    title="Spindle null finding in insomnia",
    abstract="Spindle density showed no significant association with insomnia severity. Beta power was increased in insomnia patients.",
)
direction_evidence = extract_evidence([direction_paper], rules_path=RULES_CONFIG)
for e in direction_evidence:
    print(e.mechanism, e.direction.value, e.effect_direction, "|", e.source_text)

## 10. Animal / translational evidence check

In [ ]:
animal_paper = LiteratureRecord(
    paper_id="animal_demo",
    title="Orexin optogenetic sleep circuit study",
    abstract="Optogenetic stimulation of orexin neurons in mice increased wakefulness and altered NREM sleep transitions.",
    year=2024,
    journal="Journal of Sleep Mechanisms",
    citation_count=8,
    citation_source="mock",
    citation_count_age_normalized=8 / 3,
)
animal_evidence = extract_evidence([animal_paper], rules_path=RULES_CONFIG)
for e in animal_evidence:
    print({
        "mechanism": e.mechanism,
        "species": e.species,
        "model_system": e.model_system,
        "context": e.evidence_context,
        "downstream_role": e.downstream_role,
        "translational_relevance": e.translational_relevance,
        "translational_risk": e.translational_risk,
    })

## 11. Optional LLM mock verifier

这里是 mock，不调用真实 API。LLM 只能基于已有 `source_text` 修正字段。

In [ ]:
from sleep_ai_scientist.llm.evidence_verifier import EvidenceVerifier

class MockLLMClient:
    def complete(self, prompt: str) -> str:
        return json.dumps({
            "verified_claims": [
                {
                    "claim": "Spindle density showed no significant association with insomnia severity.",
                    "mechanism": "spindle generation",
                    "population": "insomnia",
                    "condition": "insomnia",
                    "comparison_group": None,
                    "modality": "EEG",
                    "variable_or_feature": "spindle_density",
                    "direction": "null",
                    "effect_direction": "no_difference",
                    "evidence_type": "empirical",
                    "study_design": "unknown",
                    "sample_size_total": None,
                    "species": "human",
                    "limitations": [],
                    "confounds": [],
                    "statistical_note": "no significant association",
                    "confidence_reason": "Mock correction from source_text negation.",
                    "should_include": True,
                    "warnings": []
                }
            ]
        })

mock_llm_config = {
    "evidence_extraction": {"llm_assist_enabled": True},
}
verifier = EvidenceVerifier(client=MockLLMClient(), fail_open=True)
llm_checked = extract_evidence(
    [direction_paper],
    rules_path=RULES_CONFIG,
    llm_verifier=verifier,
    llm_config=mock_llm_config,
)
print("calls_attempted =", verifier.calls_attempted)
for e in llm_checked:
    print(e.mechanism, e.direction.value, e.llm_verified, e.llm_revision_applied, e.confidence_reason)

## 12. Grade evidence

In [ ]:
from sleep_ai_scientist.grounding.evidence_grader import grade_evidence_records, evidence_quality_summary

graded = grade_evidence_records(raw_evidence + animal_evidence + direction_evidence)
summary = evidence_quality_summary(graded)
print(json.dumps(summary, indent=2))

for e in graded[:8]:
    print(e.evidence_id, e.mechanism, {
        "extraction": e.extraction_confidence_score,
        "quality": e.evidence_quality_score,
        "mechanistic": e.mechanistic_strength_score,
        "clinical": e.clinical_applicability_score,
        "feasibility": e.evidence_feasibility_score,
        "final": e.final_evidence_score,
    })

## 13. Audit and benchmark in a scratch directory

In [ ]:
from sleep_ai_scientist.grounding.evidence_audit import build_evidence_audit, write_evidence_audit
from sleep_ai_scientist.grounding.evidence_benchmark import run_evidence_benchmark

scratch = ROOT / "outputs/grounding/notebook_debug"
scratch.mkdir(parents=True, exist_ok=True)

llm_stats = {
    "enabled": False,
    "calls_attempted": 0,
    "calls_succeeded": 0,
    "calls_failed": 0,
    "revised_evidence_count": 0,
    "split_claim_count": 0,
    "excluded_claim_count": 0,
}
preferred = config.get("evidence", {}).get("preferred_mechanisms", [])
audit = build_evidence_audit(papers + [animal_paper, direction_paper], graded, preferred, llm_stats)
write_evidence_audit(scratch, audit, papers + [animal_paper, direction_paper], graded)
print(json.dumps({k: audit[k] for k in ["evidence_count", "species_counts", "evidence_context_counts", "mean_final_evidence_score"]}, indent=2))

benchmark = run_evidence_benchmark(
    graded,
    ROOT / "data/fixtures/evidence_goldset/gold_evidence.csv",
    scratch / "evidence_extraction_benchmark.json",
)
print(json.dumps(benchmark, indent=2))

## 14. Variable mapping sanity check

Only analysis-ready variables can enter approved variables.

In [ ]:
from sleep_ai_scientist.grounding.data_profile import build_observed_profile, build_analysis_ready_profile
from sleep_ai_scientist.grounding.variable_mapper import map_variables

observed = build_observed_profile(config)
analysis_ready = build_analysis_ready_profile(config, observed)
ready_names = {f.feature_name for f in analysis_ready.features}
mappings = map_variables(graded, analysis_ready, config_path(config, "variable_mapping_rules"))

approved = sorted({name for m in mappings for name in m.approved_data_features})
print("analysis_ready_features =", len(ready_names))
print("approved =", approved)
print("hallucinated =", sorted(set(approved) - ready_names))
for m in mappings:
    print(m.concept, m.mapping_status.value, m.approved_data_features, m.candidate_variables)

## 15. Full rule-only grounding build

This writes the normal Phase 1 grounding artifacts under `outputs/grounding` and reports. Run only when you want to refresh outputs.

In [ ]:
RUN_PIPELINE = False

if RUN_PIPELINE:
    from sleep_ai_scientist.grounding.grounding_pipeline import run_grounding_pipeline

    result = run_grounding_pipeline(
        ROOT / "configs/grounding_config.yaml",
        corpus_version="sleepagent_grounding_corpus_v1",
    )
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("Set RUN_PIPELINE = True to run the full offline grounding build.")

## 16. Inspect generated artifacts

In [ ]:
artifact_paths = [
    ROOT / "outputs/grounding/evidence_table.csv",
    ROOT / "outputs/grounding/evidence_table.json",
    ROOT / "outputs/grounding/evidence_quality_summary.json",
    ROOT / "outputs/grounding/evidence_extraction_audit.json",
    ROOT / "outputs/grounding/evidence_extraction_benchmark.json",
    ROOT / "outputs/grounding/mechanism_graph.json",
    ROOT / "outputs/grounding/evidence_to_variable_map.yaml",
    ROOT / "outputs/grounding/corpus_manifest.json",
    ROOT / "reports/phase1_grounding_report.md",
]
for path in artifact_paths:
    print(path.relative_to(ROOT), "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)

In [ ]:
report_path = ROOT / "reports/phase1_grounding_report.md"
if report_path.exists():
    report = report_path.read_text(encoding="utf-8")
    for section in [
        "## Evidence Extraction Completeness",
        "## Evidence Quality and Feasibility",
        "## Citation and Journal Metadata",
        "## Animal and Translational Evidence",
        "## LLM-assisted Evidence Verification",
        "## Evidence Extraction Benchmark",
    ]:
        print(section, "->", section in report)